In [2]:
# =================================================================
# NOTEBOOK: 03_modeling.ipynb
# PHASE 4: MODELING (Baseline, Bagging, Boosting)
# GOAL: Preprocess, select features, train 3 models, and print the specific accuracy percentage for each.
# =================================================================

import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. File Paths and Setup ---
BASE_PATH = 'C:\\Users\\acer\\OneDrive\\Desktop\\AimlWebforecasting'
DATA_INPUT_PATH = f'{BASE_PATH}\\data\\processed\\features_df.pkl'
ASSETS_PATH = f'{BASE_PATH}\\assets'
RESULTS_PATH = f'{BASE_PATH}\\results'

os.makedirs(ASSETS_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load the engineered feature matrix
try:
    df = pd.read_pickle(DATA_INPUT_PATH)
    print("--- Feature matrix loaded successfully ---")
except FileNotFoundError:
    print(f"Error: File not found at {DATA_INPUT_PATH}. Run 02_feature_engineering.ipynb first.")
    exit()

# Separate features (X) and target (y)
y = df['y']
X = df.drop(columns=['y'])

# --- 2. Data Splitting (Time-Based) ---
TEST_SIZE = 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False
)

print(f"\nTrain set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")


# --- 3. Preprocessing Pipeline Definition ---
categorical_features = ['dow', 'month', 'week_of_year', 'is_weekend', 'promo_flag', 'holiday_flag']
numerical_features = X.columns.drop(categorical_features, errors='ignore').tolist()

# Define the preprocessor: Scale numerical, One-Hot Encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='drop'
)

# Fit and transform the training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names_processed = preprocessor.get_feature_names_out()
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names_processed)


# --- 4. Dimensionality Reduction (Feature Importance) ---

# Train a temporary RF model for feature ranking
temp_rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10, n_jobs=-1)
temp_rf_model.fit(X_train_processed_df, y_train)

# Select the top N features
TOP_N = 15
importance = pd.Series(temp_rf_model.feature_importances_, index=feature_names_processed)
importance_sorted = importance.sort_values(ascending=False)
selected_features = importance_sorted.head(TOP_N).index.tolist()

print(f"\n--- Top {TOP_N} Selected Features ---")
# Plot and save feature importance plot
plt.figure(figsize=(10, 8))
importance_sorted.head(TOP_N).sort_values(ascending=True).plot(kind='barh')
plt.title(f'Top {TOP_N} Feature Importance (Random Forest)')
plt.xlabel('Feature Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}\\feature_importance.png')
plt.close()
print(f"Feature importance plot saved to '{RESULTS_PATH}\\feature_importance.png'.")

# Filter the processed data using the selected features
X_train_final = X_train_processed_df[selected_features]
X_test_final = pd.DataFrame(X_test_processed, columns=feature_names_processed)[selected_features]

# Save the necessary assets for the Streamlit app
joblib.dump(preprocessor, f'{ASSETS_PATH}\\preprocessor.pkl')
np.save(f'{ASSETS_PATH}\\selected_features.npy', selected_features)
print("\nPreprocessor and selected features saved to 'assets/' folder.")

# --- 5. Model Training and Evaluation Utility ---

def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Trains and evaluates a model, returning R2 (Accuracy), MAE, and RMSE."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"\n--- {model_name} Performance ---")
    
    # MODIFIED OUTPUT TO MATCH REQUESTED FORMAT
    print(f"The accuracy for the {model_name} model is {r2 * 100:.2f} percentage.") 
    
    print(f"MAE: ${mae:.2f}")
    
    return model, r2

# --- 6. Model 1: Linear Regression (Baseline) ---
print("\n" + "="*70)
lr_model = LinearRegression()
lr_model, lr_r2 = evaluate_model(lr_model, X_train_final, y_train, X_test_final, y_test, "Linear Regression (Baseline)")


# --- 7. Model 2: Random Forest Regressor (Bagging) ---
rf_model = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
rf_model, rf_r2 = evaluate_model(rf_model, X_train_final, y_train, X_test_final, y_test, "Random Forest (Bagging)")


# --- 8. Model 3: XGBoost Regressor (Boosting & Tuning) ---
print("\n--- Starting XGBoost Training and Tuning ---")

# Define parameter space for Random Search
param_dist = {
    'n_estimators': [300, 500],
    'learning_rate': [0.05, 0.1],
    'max_depth': [5, 7],
    'subsample': [0.7, 0.9],
}

xgb_base = xgb.XGBRegressor(random_state=42, tree_method='hist')
random_search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=param_dist, 
    n_iter=10, 
    scoring='r2', 
    cv=3, 
    verbose=0, 
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_final, y_train)
xgb_best_model = random_search.best_estimator_

# Evaluate the best XGBoost model
xgb_best_model, xgb_r2 = evaluate_model(xgb_best_model, X_train_final, y_train, X_test_final, y_test, "XGBoost (Boosting + Tuned)")

print("="*70)

# --- 9. Save the Final Model ---
# Save the best performing model (XGBoost is assumed best)
joblib.dump(xgb_best_model, f'{RESULTS_PATH}\\final_xgb_model.pkl')
print(f"\nFinal XGBoost model saved to '{RESULTS_PATH}\\final_xgb_model.pkl'.")

# =================================================================
# END OF NOTEBOOK 03 - Proceed to 04_evaluation_and_viz.ipynb
# =================================================================

--- Feature matrix loaded successfully ---

Train set size: 1176 rows
Test set size: 294 rows

--- Top 15 Selected Features ---
Feature importance plot saved to 'C:\Users\acer\OneDrive\Desktop\AimlWebforecasting\results\feature_importance.png'.

Preprocessor and selected features saved to 'assets/' folder.


--- Linear Regression (Baseline) Performance ---
The accuracy for the Linear Regression (Baseline) model is 96.69 percentage.
MAE: $108.22

--- Random Forest (Bagging) Performance ---
The accuracy for the Random Forest (Bagging) model is 94.27 percentage.
MAE: $139.07

--- Starting XGBoost Training and Tuning ---

--- XGBoost (Boosting + Tuned) Performance ---
The accuracy for the XGBoost (Boosting + Tuned) model is 93.51 percentage.
MAE: $147.06

Final XGBoost model saved to 'C:\Users\acer\OneDrive\Desktop\AimlWebforecasting\results\final_xgb_model.pkl'.
